# Importación de librerías (XGBoost)

In [1]:
# ==========================================
# CELDA 1: LIBRERÍAS (CP-08)
# ==========================================
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import ADASYN

df_info = pd.read_csv('./../../dataset/oulad/studentInfo.csv')
df_vle = pd.read_csv('./../../dataset/oulad/studentVle.csv')

df_info['target'] = df_info['final_result'].apply(lambda x: 1 if x == 'Withdrawn' else 0)
presentaciones_2013 = ['2013B', '2013J']
presentaciones_2014 = ['2014B', '2014J']
df_train_info = df_info[df_info['code_presentation'].isin(presentaciones_2013)].copy()
df_test_info = df_info[df_info['code_presentation'].isin(presentaciones_2014)].copy()

# Función de preparación y bucle de entrenamiento

In [5]:
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import ADASYN

class ThresholdWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self

    def predict(self, X):
        proba = self.estimator.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

scoring_metrics = {
    'Accuracy': make_scorer(accuracy_score),
    'Precision': make_scorer(precision_score, zero_division=0),
    'Recall': make_scorer(recall_score, zero_division=0),
    'F1-Score': make_scorer(f1_score, zero_division=0)
}

param_grid_xgb = {
    'clasificador__estimator__max_depth': [3, 5, 7],
    'clasificador__estimator__learning_rate': [0.01, 0.1],
    'clasificador__threshold': [0.3, 0.4, 0.5, 0.6]
}

ventanas_temporales = [30, 60, 90]
estrategias_balanceo = ['Línea Base (Sin balanceo)', 'Undersampling', 'ADASYN']

resultados_mejores = []
todas_las_permutaciones = []

print("Iniciando búsqueda exhaustiva CP-08 (XGBoost)...")

for dias in ventanas_temporales:
    print(f"--- Evaluando ventana a {dias} días ---")

    X_train, y_train = preparar_datos(df_vle, df_train_info, dias)
    X_test, y_test = preparar_datos(df_vle, df_test_info, dias)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for estrategia in estrategias_balanceo:
        if estrategia == 'Undersampling':
            sampler = RandomUnderSampler(random_state=42)
        elif estrategia == 'ADASYN':
            sampler = ADASYN(random_state=42, n_neighbors=5)
        else:
            sampler = None

        pasos = []
        if sampler is not None:
            pasos.append(('sampler', sampler))

        xgb_base = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
        wrapper = ThresholdWrapper(estimator=xgb_base)
        pasos.append(('clasificador', wrapper))

        pipeline = ImbPipeline(pasos)

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid_xgb,
            cv=5,
            scoring=scoring_metrics,
            refit='Recall',
            n_jobs=-1,
            return_train_score=False
        )

        grid_search.fit(X_train_scaled, y_train)

        mejor_modelo = grid_search.best_estimator_
        y_pred = mejor_modelo.predict(X_test_scaled)

        resultados_mejores.append({
            'Días': dias,
            'Estrategia': estrategia,
            'Mejor Max_Depth': grid_search.best_params_['clasificador__estimator__max_depth'],
            'Mejor Learning_Rate': grid_search.best_params_['clasificador__estimator__learning_rate'],
            'Mejor Threshold': grid_search.best_params_['clasificador__threshold'],
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1-Score': f1_score(y_test, y_pred, zero_division=0)
        })

        cv_results = grid_search.cv_results_
        for i in range(len(cv_results['params'])):
            todas_las_permutaciones.append({
                'Días': dias,
                'Estrategia': estrategia,
                'Max_Depth': cv_results['param_clasificador__estimator__max_depth'][i],
                'Learning_Rate': cv_results['param_clasificador__estimator__learning_rate'][i],
                'Threshold': cv_results['param_clasificador__threshold'][i],
                'CV_Accuracy': cv_results['mean_test_Accuracy'][i],
                'CV_Precision': cv_results['mean_test_Precision'][i],
                'CV_Recall': cv_results['mean_test_Recall'][i],
                'CV_F1': cv_results['mean_test_F1-Score'][i]
            })

        print(f"  > {estrategia}: Procesado.")

df_mejores = pd.DataFrame(resultados_mejores)
df_todas_permutaciones = pd.DataFrame(todas_las_permutaciones)

mejor_fila = df_mejores.sort_values(by=['Recall', 'F1-Score'], ascending=[False, False]).iloc[0]
mejor_dia = mejor_fila['Días']
mejor_estrategia = mejor_fila['Estrategia']
mejor_depth = mejor_fila['Mejor Max_Depth']
mejor_lr = mejor_fila['Mejor Learning_Rate']

df_tradeoff = df_todas_permutaciones[
    (df_todas_permutaciones['Días'] == mejor_dia) &
    (df_todas_permutaciones['Estrategia'] == mejor_estrategia) &
    (df_todas_permutaciones['Max_Depth'] == mejor_depth) &
    (df_todas_permutaciones['Learning_Rate'] == mejor_lr)
].copy()

df_tradeoff = df_tradeoff.sort_values(by=['Threshold'])

print("\n=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===")
display(df_mejores)

print(f"\n=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO ({mejor_dia} Días | {mejor_estrategia} | Depth={mejor_depth} | LR={mejor_lr}) ===")
display(df_tradeoff)

Iniciando búsqueda exhaustiva CP-08 (XGBoost)...
--- Evaluando ventana a 30 días ---


C:\Users\joseb\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:58:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  > Línea Base (Sin balanceo): Procesado.


C:\Users\joseb\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:58:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  > Undersampling: Procesado.


...

  > Undersampling: Procesado.
  > ADASYN: Procesado.

=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===


C:\Users\joseb\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:59:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,Días,Estrategia,Mejor Max_Depth,Mejor Learning_Rate,Mejor Threshold,Accuracy,Precision,Recall,F1-Score
0,30,Línea Base (Sin balanceo),5,0.10,0.3,0.718370,0.602334,0.489051,0.539813
1,30,Undersampling,3,0.01,0.3,0.337757,0.337757,1.000000,0.504960
2,30,ADASYN,3,0.01,0.3,0.337757,0.337757,1.000000,0.504960
3,60,Línea Base (Sin balanceo),5,0.10,0.3,0.731274,0.623175,0.517006,0.565147
4,60,Undersampling,3,0.01,0.3,0.337757,0.337757,1.000000,0.504960
5,60,ADASYN,3,0.01,0.3,0.337757,0.337757,1.000000,0.504960
6,90,Línea Base (Sin balanceo),5,0.10,0.3,0.741765,0.636577,0.548688,0.589374
7,90,Undersampling,3,0.01,0.3,0.337757,0.337757,1.000000,0.504960
8,90,ADASYN,3,0.01,0.3,0.337757,0.337757,1.000000,0.504960



=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO (30 Días | Undersampling | Depth=3 | LR=0.01) ===


,Días,Estrategia,Max_Depth,Learning_Rate,Threshold,CV_Accuracy,CV_Precision,CV_Recall,CV_F1
24,30,Undersampling,3,0.01,0.3,0.274739,0.274739,1.000000,0.429633
25,30,Undersampling,3,0.01,0.4,0.399588,0.301138,0.897984,0.449915
26,30,Undersampling,3,0.01,0.5,0.720740,0.537882,0.506826,0.500863
27,30,Undersampling,3,0.01,0.6,0.778771,0.745782,0.317907,0.436611
